In [10]:
name = "aaa"
depth = 3
wikipedia = True
print(f"./results/{name}/depth_{depth}{'_wiki' if wikipedia else ''}.pkl")

./results/aaa/depth_3_wiki.pkl


In [24]:
def linear_function(depth):
    a = (0.0002 - 0.001) / 3
    b = 0.001 - a
    return a * depth + b

# Exemple d'utilisation
depth = 4
value = linear_function(depth)
print(f"Pour une profondeur de {depth}, la valeur est {value}")

Pour une profondeur de 4, la valeur est 0.0002000000000000001


In [5]:
import torch

def clamp_10051(tensor):
    # Mappe chaque élément sur l'un des 4 niveaux : -1, 0, 0.5, 1
    return torch.where(tensor < -0.5, torch.tensor(-1.0, dtype=tensor.dtype),
               torch.where(tensor < 0.25, torch.tensor(0.0, dtype=tensor.dtype),
                   torch.where(tensor < 0.75, torch.tensor(0.5, dtype=tensor.dtype),
                       torch.tensor(1.0, dtype=tensor.dtype)
                   )
               )
           )

def clamp_0505(tensor):
    # Renvoie -0.5 si l'élément est négatif, sinon 0.5.
    return torch.where(tensor < 0, torch.tensor(-0.5, dtype=tensor.dtype), 
                              torch.tensor(0.5, dtype=tensor.dtype))

def clamp_10(tensor):
    # Renvoie -1 si l'élément est négatif, sinon 0.
    return torch.where(tensor < 0, torch.tensor(-1, dtype=tensor.dtype), 
                              torch.tensor(0, dtype=tensor.dtype))

# Exemple d'utilisation
t = torch.Tensor([-0.02, 0.01, -10.2, 10.2, 0.65])

print("clamp_0505(t) :", clamp_0505(t))
print("clamp_10(t)  :", clamp_10(t))
print("clamp_10051(t):", clamp_10051(t))

clamp_0505(t) : tensor([-0.5000,  0.5000, -0.5000,  0.5000,  0.5000])
clamp_10(t)  : tensor([-1.,  0., -1.,  0.,  0.])
clamp_10051(t): tensor([ 0.0000,  0.0000, -1.0000,  1.0000,  0.5000])


In [16]:
from math import sqrt

print(sqrt(128*8))
print(sqrt(512*8))

32.0
64.0


In [26]:
print(sqrt(4096))
print(sqrt(16384))

print(sqrt(512*8/2))

64.0
128.0
45.254833995939045


In [ ]:
depth = 16
dim = 128 * depth
ffn_dim = 512 * depth
heads = depth
lora = 4
# vocab_size = 128000
vocab_size = 10000
length = 5
embeddings = vocab_size * dim
# embeddings = length*dim
positionnal_emb = length * dim

k = dim * dim / lora
q = dim * dim / lora
v = dim * dim / lora
o = dim * dim / lora
norm = 2 * dim
attention = heads * (k + q + v + o) + norm

ff1 = dim * dim * lora + dim * lora
ff2 = dim * dim * lora + dim
norm = 2 * dim
ffn = ff1 + ff2 + norm

transformer = depth * (attention + ffn) + embeddings + 0 * positionnal_emb
print(transformer / 10**6)

144.531456


In [2]:
from datasets import load_dataset

/home/alan/Documents/SuperQuantization/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds_train = load_dataset("codeparrot/codeparrot-train-v2-near-dedup", split="train[:10]")
ds_valid = load_dataset("codeparrot/codeparrot-valid-v2-near-dedup", split="train[:10]")

In [1]:
# import os
# os.chdir(os.path.abspath(os.path.join(os.getcwd(), '../../')))
# print(os.getcwd())
import torch
from tests.text_generation.transformer import Transformer
from super_quantization.super_quantizer import SuperQuantizer

vocab_size = 10000
seq_length = 256
depth = 1
model = Transformer(
    d_model=32*depth,
    # d_model=64,
    # d_model=4*depth,
    n_heads=depth,
    # n_heads=2,
    d_ff=128*depth,
    # d_ff=256*depth,
    # d_ff=16*depth,
    depth=depth,
    vocab_size=vocab_size,
    max_context_size=seq_length,
    lora_ratio=4
)

import collections

def analyze_model_parameters(model):
    param_groups = collections.defaultdict(float)
    total_params = 0
    
    for name, param in model.named_parameters():
        param_size = param.numel()
        total_params += param_size
        
        if "self_attention.Q" in name:
            key = "Q_weights" if "weight" in name else "Q_bias"
        elif "self_attention.K" in name:
            key = "K_weights" if "weight" in name else "K_bias"
        elif "self_attention.V" in name:
            key = "V_weights" if "weight" in name else "V_bias"
        elif "self_attention.O" in name:
            key = "O_weights" if "weight" in name else "O_bias"
        elif "fc_1" in name:
            key = "fc_1_weights" if "weight" in name else "fc_1_bias"
        elif "fc_2" in name:
            key = "fc_2_weights" if "weight" in name else "fc_2_bias"
        elif "layer_norm" in name:
            key = "LayerNorm_weights" if "weight" in name else "LayerNorm_bias"
        elif "embedding.weight" in name:
            key = "embedding.weight"
        else:
            key = name
        
        param_groups[key] += param_size

    param_distribution = {k: (v / total_params) * 100 for k, v in param_groups.items()}
    active_param_distribution = {k: (v / (total_params - param_groups["embedding.weight"])) * 100 for k, v in param_groups.items()}
    
    for param_type, percentage in param_distribution.items():
        print(f"{param_type}: {percentage:.2f}%")
    
    print()
    
    for param_type, percentage in active_param_distribution.items():
        if param_type != "embedding.weight":
            print(f"Active {param_type}: {percentage:.2f}%")
    
    return param_distribution, active_param_distribution

# Exemple d'utilisation avec un modèle fictif
analyze_model_parameters(model)
# sq = SuperQuantizer()
# print("Params for base:", sum(p.numel() for p in model.parameters() if p.requires_grad))
# print(sq.mesure(model)/32)
# sq.quantize(model, {"K": "_1012", "Q": "_1012", "O": "_1012", "V": "_1012", "fc_1": "01", "fc_2": "01"})
# print(sq.mesure(model)/32)

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7b1538837d50>>
Traceback (most recent call last):
  File "/home/alan/Documents/SuperQuantization/.venv/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


embedding.weight: 97.12%
Q_weights: 0.08%
K_weights: 0.08%
V_weights: 0.08%
O_weights: 0.08%
fc_1_weights: 1.24%
fc_1_bias: 0.04%
fc_2_weights: 1.24%
fc_2_bias: 0.01%
LayerNorm_weights: 0.02%
LayerNorm_bias: 0.02%

Active Q_weights: 2.69%
Active K_weights: 2.69%
Active V_weights: 2.69%
Active O_weights: 2.69%
Active fc_1_weights: 43.10%
Active fc_1_bias: 1.35%
Active fc_2_weights: 43.10%
Active fc_2_bias: 0.34%
Active LayerNorm_weights: 0.67%
Active LayerNorm_bias: 0.67%


({'embedding.weight': 97.11594484365897,
  'Q_weights': 0.07768498737618955,
  'K_weights': 0.07768498737618955,
  'V_weights': 0.07768498737618955,
  'O_weights': 0.07768498737618955,
  'fc_1_weights': 1.2429597980190328,
  'fc_1_bias': 0.038842493688094774,
  'fc_2_weights': 1.2429597980190328,
  'fc_2_bias': 0.009710623422023694,
  'LayerNorm_weights': 0.019421246844047387,
  'LayerNorm_bias': 0.019421246844047387},
 {'embedding.weight': 3367.340067340067,
  'Q_weights': 2.6936026936026933,
  'K_weights': 2.6936026936026933,
  'V_weights': 2.6936026936026933,
  'O_weights': 2.6936026936026933,
  'fc_1_weights': 43.09764309764309,
  'fc_1_bias': 1.3468013468013467,
  'fc_2_weights': 43.09764309764309,
  'fc_2_bias': 0.33670033670033667,
  'LayerNorm_weights': 0.6734006734006733,
  'LayerNorm_bias': 0.6734006734006733})

In [7]:
import torch
from tests.text_generation.transformer import Transformer
from super_quantization.super_quantizer import SuperQuantizer

vocab_size = 100
seq_length = 4
depth = 1
model = Transformer(
    d_model=10,
    n_heads=3,
    d_ff=3,
    depth=2,
    vocab_size=vocab_size,
    max_context_size=seq_length,
    lora_ratio=4,
    rope=True
)

x = torch.Tensor([
    [1, 2, 3, 4]
])

model(x).shape

torch.Size([1, 101, 4])

In [5]:
from datasets import load_dataset

validation_texts = load_dataset(
    "BEE-spoke-data/wikipedia-deduped", "text-only", split="test[:500000]"
)["text"]
# train_texts = load_dataset(
#     "BEE-spoke-data/wikipedia-deduped", "text-only", split="train[:500000]"
# )["text"]

Generating test split: 100%|██████████| 149300/149300 [00:17<00:00, 8567.66 examples/s] 
